# House-price CNN project — streamlined notebook scaffold

This notebook contains the **essential code workflow**, not a complete implementation. Complete each `TODO` in order. It keeps only the comparisons needed to answer the proposal:

1. Standard VGG16 spatial representation.
2. VGG16 with global average pooling (GAP).
3. GAP plus mask-guided house, surroundings, and background features.
4. The same regional model with photometric augmentation.

Removed from the longer version: extensive dataset auditing, a separate mask-statistics study, parameter-count experiments, a five-level lighting sweep, and detailed residual/error-case testing.

## 1. Imports and fixed settings

All experiments must use the same split, VGG16 weights, optimiser settings, loss, and evaluation metrics. Only the representation or augmentation condition should change. Update the three dataset column names after inspecting your metadata file.

In [26]:
%pip install torch torchvision transformers

Note: you may need to restart the kernel to use updated packages.


In [28]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageEnhance

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from transformers import (
    AutoImageProcessor,
    SegformerForSemanticSegmentation
)

CONFIG = {
    "seed": 48,

    # Actual train.csv column names
    "id_column": "imageid",
    "target_column": "price",
    "image_column": "image_path",

    "image_size": 224,
    "batch_size": 32,
    "num_workers": 0,

    "train_fraction": 0.70,
    "validation_fraction": 0.15,
    "test_fraction": 0.15,

    "projection_dim": 128,
    "dropout": 0.30,

    "epochs": 20,
    "learning_rate_head": 1e-3,
    "learning_rate_backbone": 1e-5,
    "weight_decay": 1e-4,
    "early_stopping_patience": 4,
    "use_log1p_target": False,

    "segmentation_model":
        "nvidia/segformer-b0-finetuned-ade-512-512",

    "surroundings_dilation_pixels": 32,

    "brightness_jitter": 0.30,
    "contrast_jitter": 0.30,
    "saturation_jitter": 0.20,
    "hue_jitter": 0.05,

    "brightness_test_factors": [
        0.75,
        1.00,
        1.25
    ],
}

PATHS = {
    # CSV files
    "train_csv": Path("house_dataset/train.csv"),
    "test_csv": Path("house_dataset/test.csv"),
    "sample_solution": Path("house_dataset/sample_solution.csv"),

    # Image folders
    "train_images": Path("house_dataset/train"),
    "test_images": Path("house_dataset/test"),

    # Generated project files
    "splits": Path("artifacts/splits"),
    "masks": Path("artifacts/masks"),
    "checkpoints": Path("artifacts/checkpoints"),
    "results": Path("artifacts/results"),
}

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])

for key in [
    "splits",
    "masks",
    "checkpoints",
    "results"
]:
    PATHS[key].mkdir(
        parents=True,
        exist_ok=True
    )

print("Working directory:", Path.cwd())
print("Device:", DEVICE)

Working directory: C:\Users\Ning\COMP90086_Final_Project
Device: cpu


## 2. Load metadata and create one fixed split

Perform only essential checks: required columns, positive prices, existing image files, and no property overlap between splits. If a property has several photographs, all its photographs must remain in one split. Save the split CSV files and reuse them in every experiment.

In [30]:
# Load CSV files

train_df = pd.read_csv(
    PATHS["train_csv"],
    dtype={"imageid": str}
)

test_df = pd.read_csv(
    PATHS["test_csv"],
    dtype={"imageid": str}
)

sample_solution_df = pd.read_csv(
    PATHS["sample_solution"],
    dtype={"imageid": str}
)


# Column names

ID_COLUMN = CONFIG["id_column"]
TARGET_COLUMN = CONFIG["target_column"]
IMAGE_COLUMN = CONFIG["image_column"]


# Connect each filename to the training image folder

train_df[IMAGE_COLUMN] = train_df[ID_COLUMN].apply(
    lambda filename:
        PATHS["train_images"] / filename
)


# Connect each filename to the test image folder

test_df[IMAGE_COLUMN] = test_df[ID_COLUMN].apply(
    lambda filename:
        PATHS["test_images"] / filename
)


# Display the result

display(
    train_df[
        [
            ID_COLUMN,
            TARGET_COLUMN,
            IMAGE_COLUMN
        ]
    ].head()
)

,imageid,price,image_path
0,1.jpg,549.9,house_dataset\train\1.jpg
1,2.jpg,265.0,house_dataset\train\2.jpg
2,3.jpg,895.0,house_dataset\train\3.jpg
3,4.jpg,650.0,house_dataset\train\4.jpg
4,5.jpg,1799.0,house_dataset\train\5.jpg


In [38]:
# Confirm that the requested fractions add up to 100%
total_fraction = (
    CONFIG["train_fraction"]
    + CONFIG["validation_fraction"]
    + CONFIG["test_fraction"]
)

assert np.isclose(total_fraction, 1.0), (
    "train_fraction, validation_fraction, and test_fraction "
    "must add up to 1.0."
)


development_df, internal_test_df = train_test_split(
    train_df,
    test_size=CONFIG["test_fraction"],
    random_state=CONFIG["seed"],
    shuffle=True
)


# The validation fraction must be calculated relative to the
# remaining development data.
validation_share_of_development = (
    CONFIG["validation_fraction"]
    / (
        CONFIG["train_fraction"]
        + CONFIG["validation_fraction"]
    )
)

train_split_df, validation_df = train_test_split(
    development_df,
    test_size=validation_share_of_development,
    random_state=CONFIG["seed"],
    shuffle=True
)

train_split_df = train_split_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
internal_test_df = internal_test_df.reset_index(drop=True)

train_ids = set(train_split_df[CONFIG["id_column"]])
validation_ids = set(validation_df[CONFIG["id_column"]])
internal_test_ids = set(internal_test_df[CONFIG["id_column"]])

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(internal_test_ids)
assert validation_ids.isdisjoint(internal_test_ids)

assert (
    len(train_split_df)
    + len(validation_df)
    + len(internal_test_df)
    == len(train_df)
)


train_split_df.to_csv(
    PATHS["splits"] / "train_split.csv",
    index=False
)

validation_df.to_csv(
    PATHS["splits"] / "validation_split.csv",
    index=False
)

internal_test_df.to_csv(
    PATHS["splits"] / "internal_test_split.csv",
    index=False
)

TARGET_MEAN = float(
    train_split_df[CONFIG["target_column"]].mean()
)

TARGET_STD = float(
    train_split_df[CONFIG["target_column"]].std()
)

print(train_split_df.shape)
print(validation_df.shape)
print(internal_test_df.shape)
print(TARGET_MEAN, TARGET_STD)

(5600, 3)
(1200, 3)
(1200, 3)
721.7907048214286 395.00169466521675


## 3. Image transformations

Use ordinary ImageNet preprocessing for E1–E3. Add colour and lighting jitter only for E4. Validation and test transforms must be deterministic. When implementing the dataset, make sure random resizing/flipping is shared by the image and masks, while colour jitter is applied only to the image.

In [40]:
# Normalization values used when VGG16 was trained on ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


standard_train_transform = transforms.Compose([
    transforms.Resize(
        (CONFIG["image_size"], CONFIG["image_size"])
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


photometric_train_transform = transforms.Compose([
    transforms.Resize(
        (CONFIG["image_size"], CONFIG["image_size"])
    ),
    transforms.ColorJitter(
        brightness=CONFIG["brightness_jitter"],
        contrast=CONFIG["contrast_jitter"],
        saturation=CONFIG["saturation_jitter"],
        hue=CONFIG["hue_jitter"]
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

evaluation_transform = transforms.Compose([
    transforms.Resize(
        (CONFIG["image_size"], CONFIG["image_size"])
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


## 4. Dataset and DataLoaders

Return the image, price, property ID, and—only when requested—the three masks. Use `log1p(price)` during training if configured, then transform predictions back with `expm1` before calculating metrics.

In [42]:
class HousePriceDataset(Dataset):
    """
    Connects each house image with its corresponding price.

    Each returned sample contains:
        image: processed image tensor
        target: house price or log-transformed house price
        image_id: filename of the image
    """

    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        # Select one row from the DataFrame
        row = self.dataframe.iloc[index]

        # Read the image
        image_path = row[CONFIG["image_column"]]
        image = Image.open(image_path).convert("RGB")

        # Apply resizing, tensor conversion and normalization
        image = self.transform(image)

        # Read the corresponding house price
        price = float(row[CONFIG["target_column"]])
        
        standardized_price = (price - TARGET_MEAN) / TARGET_STD
        
        target = torch.tensor(standardized_price, dtype=torch.float32)

        image_id = str(row[CONFIG["id_column"]])

        return {
            "image": image,
            "target": target,
            "image_id": image_id
        }


# ============================================================
# Create datasets
# ============================================================

# Used for Models E1, E2 and E3
standard_train_dataset = HousePriceDataset(
    dataframe=train_split_df,
    transform=standard_train_transform
)

# Used for Model E4
photometric_train_dataset = HousePriceDataset(
    dataframe=train_split_df,
    transform=photometric_train_transform
)

# Used for validation
validation_dataset = HousePriceDataset(
    dataframe=validation_df,
    transform=evaluation_transform
)

# Used for the final internal model comparison
internal_test_dataset = HousePriceDataset(
    dataframe=internal_test_df,
    transform=evaluation_transform
)


# ============================================================
# Create DataLoaders
# ============================================================

standard_train_loader = DataLoader(
    standard_train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available()
)

photometric_train_loader = DataLoader(
    photometric_train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available()
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available()
)

internal_test_loader = DataLoader(
    internal_test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available()
)

print("Datasets and DataLoaders created successfully.")
print(f"Standard training batches:    {len(standard_train_loader)}")
print(f"Photometric training batches: {len(photometric_train_loader)}")
print(f"Validation batches:           {len(validation_loader)}")
print(f"Internal test batches:        {len(internal_test_loader)}")
print("Kaggle test data has not been loaded.")

Datasets and DataLoaders created successfully.
Standard training batches:    175
Photometric training batches: 175
Validation batches:           38
Internal test batches:        38
Kaggle test data has not been loaded.


## 5. Load Standard CNN backbone

In [44]:
class StandardCNNRegressor(nn.Module):

    def __init__(self, projection_dim=128, dropout=0.30):
        super().__init__()

        vgg16 = models.vgg16(
            weights=models.VGG16_Weights.IMAGENET1K_V1
        )

        self.backbone = vgg16.features

        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

        for parameter in self.backbone[-7:].parameters():
            parameter.requires_grad = True

        self.spatial_pool = nn.AdaptiveAvgPool2d((7, 7))

        self.regression_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, projection_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(projection_dim, 1)
        )

    def forward(self, images):

        feature_maps = self.backbone(images)

        spatial_features = self.spatial_pool(feature_maps)

        predictions = self.regression_head(spatial_features)

        return predictions.squeeze(1)

set_seed(CONFIG["seed"])

# Create Model E1
model_e1 = StandardCNNRegressor(
    projection_dim=CONFIG["projection_dim"],
    dropout=CONFIG["dropout"]
).to(DEVICE)


# Display parameter information
total_parameters = sum(
    parameter.numel()
    for parameter in model_e1.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model_e1.parameters()
    if parameter.requires_grad
)

print("Model E1 created successfully.")
print(f"Device: {DEVICE}")
print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")

Model E1 created successfully.
Device: cpu
Total parameters: 17,926,209
Trainable parameters: 10,290,945


## 6. Train and evaluate Pipeline

In [46]:
def forward_batch(model, batch, device):
    """Run either an image-only model or an image-and-mask model.

    E1/E2 batches contain: image, target, image_id.
    E3/E4 batches additionally contain masks with shape [B, R, H, W].
    """
    images = batch["image"].to(device)

    if "masks" in batch:
        masks = batch["masks"].to(device)
        return model(images, masks)

    return model(images)


def train_one_epoch(model, data_loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in data_loader:
        images = batch["image"].to(device)
        targets = batch["target"].to(device)

        optimizer.zero_grad()

        predictions = forward_batch(model, batch, device)
        loss = criterion(predictions, targets)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


def evaluate_model(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    predicted_targets = []
    true_targets = []

    with torch.no_grad():
        for batch in data_loader:
            images = batch["image"].to(device)
            targets = batch["target"].to(device)

            predictions = forward_batch(model, batch, device)
            loss = criterion(predictions, targets)

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            predicted_targets.append(predictions.cpu())
            true_targets.append(targets.cpu())

    predicted_targets = torch.cat(predicted_targets).numpy()
    true_targets = torch.cat(true_targets).numpy()

    # Convert standardized targets back to original prices
    predicted_prices = (predicted_targets * TARGET_STD + TARGET_MEAN)
    
    true_prices = (true_targets * TARGET_STD + TARGET_MEAN)

    # A house price should not be negative
    predicted_prices = np.maximum(predicted_prices, 0)

    mse = mean_squared_error(true_prices, predicted_prices)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(true_prices, predicted_prices)
    r2 = r2_score(true_prices, predicted_prices)

    return {
        "loss": total_loss / total_samples,
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }


def create_optimizer(model):
    """Create the same two-learning-rate optimizer for any experiment model."""
    if not hasattr(model, "backbone"):
        raise AttributeError("Every experiment model must expose its CNN as model.backbone.")

    backbone_parameters = [
        parameter
        for parameter in model.backbone.parameters()
        if parameter.requires_grad
    ]
    backbone_parameter_ids = {id(parameter) for parameter in model.backbone.parameters()}
    head_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad and id(parameter) not in backbone_parameter_ids
    ]

    parameter_groups = []
    if backbone_parameters:
        parameter_groups.append({
            "params": backbone_parameters,
            "lr": CONFIG["learning_rate_backbone"]
        })
    if head_parameters:
        parameter_groups.append({
            "params": head_parameters,
            "lr": CONFIG["learning_rate_head"]
        })
    if not parameter_groups:
        raise ValueError("The model has no trainable parameters.")

    return torch.optim.AdamW(
        parameter_groups,
        weight_decay=CONFIG["weight_decay"]
    )


def fit_model(model, train_loader, validation_loader, model_name, device=DEVICE):
    """Train, validate, early-stop, save, and reload one experiment model."""
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = create_optimizer(model)
    checkpoint_path = PATHS["checkpoints"] / f"{model_name}.pt"
    history_path = PATHS["results"] / f"{model_name}_training_history.csv"

    history = []
    best_validation_mse = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, CONFIG["epochs"] + 1):
        training_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        
        validation_metrics = evaluate_model(
            model, validation_loader, criterion, device
        )

        history.append({
            "epoch": epoch,
            "training_loss": training_loss,
            **{f"validation_{key}": value for key, value in validation_metrics.items()}
        })

        print(
            f"{model_name} | Epoch {epoch:02d}/{CONFIG['epochs']} | "
            f"MSE: {validation_metrics['mse']:.4f} | "
            f"RMSE: {validation_metrics['rmse']:.4f} | "
            f"MAE: {validation_metrics['mae']:.4f} | "
            f"R²: {validation_metrics['r2']:.4f}"
        )

        if validation_metrics["mse"] < best_validation_mse:
            best_validation_mse = float(validation_metrics["mse"])
            epochs_without_improvement = 0
            torch.save({
                "epoch": int(epoch),
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "validation_mse": best_validation_mse,
                "target_mean": float(TARGET_MEAN),
                "target_std": float(TARGET_STD),
                "target_method": "standardization",
                "loss_function": criterion.__class__.__name__,
                "model_name": model_name
            }, checkpoint_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= CONFIG["early_stopping_patience"]:
            break

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    best_metrics = evaluate_model(model, validation_loader, criterion, device)

    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)

    summary = {
        "model": model_name,
        "best_epoch": checkpoint["epoch"],
        **{f"validation_{key}": value for key, value in best_metrics.items()},
        "checkpoint": str(checkpoint_path)
    }
    return model, history_df, summary

## 7. Training the standard CNN model

In [ ]:
# The same call is used for every model; only the model and training loader change.
model_e1, history_e1_df, summary_e1 = fit_model(
    model=model_e1,
    train_loader=standard_train_loader,
    validation_loader=validation_loader,
    model_name="model_e1_standard_cnn",
    device=DEVICE
)

pd.DataFrame([summary_e1])

model_e1_standard_cnn | Epoch 01/20 | MSE: 128737.3750 | RMSE: 358.7999 | MAE: 280.7045 | R²: 0.1620
model_e1_standard_cnn | Epoch 02/20 | MSE: 123107.5859 | RMSE: 350.8669 | MAE: 264.6642 | R²: 0.1986
model_e1_standard_cnn | Epoch 03/20 | MSE: 121780.5859 | RMSE: 348.9708 | MAE: 263.0470 | R²: 0.2073
model_e1_standard_cnn | Epoch 04/20 | MSE: 121019.6406 | RMSE: 347.8788 | MAE: 257.5700 | R²: 0.2122
model_e1_standard_cnn | Epoch 05/20 | MSE: 122876.7500 | RMSE: 350.5378 | MAE: 260.9690 | R²: 0.2001
model_e1_standard_cnn | Epoch 06/20 | MSE: 123196.1328 | RMSE: 350.9931 | MAE: 258.8021 | R²: 0.1981


### Reusing the pipeline for E2–E3

Each model uses `fit_model`; supply the loader that matches its inputs. E1 and E2 use image-only loaders. E3 and E4 use loaders that also return a `masks` tensor.

```python
# E2: model_e2, history_e2_df, summary_e2 = fit_model(
#     model_e2, standard_train_loader, validation_loader, "model_e2_global_pooling"
# )

# E3: model_e3, history_e3_df, summary_e3 = fit_model(
#     model_e3, masked_train_loader, masked_validation_loader, "model_e3_masked_regions"
# )

```